In [2]:
import os
import shutil
import random

# ==========================================
# CẤU HÌNH THÔNG SỐ (BẠN CẦN THAY ĐỔI Ở ĐÂY)
# ==========================================
SOURCE_DIR = "./dataset"         # Đường dẫn tới thư mục giải nén từ Roboflow
TARGET_DIR = "./final_dataset"       # Đường dẫn thư mục đầu ra

# Mở file data.yaml trong dataset gốc để xem ID của lớp "Brown spot" là số mấy (từ 0 đến 7)
MAJORITY_CLASS_ID = 1                # Giả sử Brown spot là class 3
RETAIN_RATIO = 0.3                   # Tỷ lệ giữ lại ảnh "chỉ có Brown spot" (0.3 = Giữ lại 30%, xóa 70%)

# ==========================================
# HÀM XỬ LÝ CHÍNH
# ==========================================
def balance_and_preprocess_yolo(src_dir, tgt_dir, majority_id, retain_ratio):
    # Tạo cấu trúc thư mục mới
    splits = ['train', 'valid', 'test']
    for split in splits:
        os.makedirs(os.path.join(tgt_dir, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(tgt_dir, split, 'labels'), exist_ok=True)

    # Copy file data.yaml
    if os.path.exists(os.path.join(src_dir, 'data.yaml')):
        shutil.copy(os.path.join(src_dir, 'data.yaml'), os.path.join(tgt_dir, 'data.yaml'))

    for split in splits:
        src_images_dir = os.path.join(src_dir, split, 'images')
        src_labels_dir = os.path.join(src_dir, split, 'labels')
        tgt_images_dir = os.path.join(tgt_dir, split, 'images')
        tgt_labels_dir = os.path.join(tgt_dir, split, 'labels')

        # Bỏ qua nếu thư mục không tồn tại
        if not os.path.exists(src_labels_dir):
            continue

        label_files = [f for f in os.listdir(src_labels_dir) if f.endswith('.txt')]

        keep_files = []
        exclusive_majority_files = []

        # Phân loại file cho tập TRAIN
        if split == 'train':
            for label_file in label_files:
                label_path = os.path.join(src_labels_dir, label_file)
                with open(label_path, 'r') as f:
                    lines = f.readlines()

                # Lấy danh sách các class ID có trong ảnh
                class_ids = set([int(line.split()[0]) for line in lines if line.strip()])

                # Nếu ảnh CHỈ CÓ duy nhất class đa số
                if len(class_ids) == 1 and majority_id in class_ids:
                    exclusive_majority_files.append(label_file)
                else:
                    keep_files.append(label_file) # Giữ lại toàn bộ ảnh có nhãn khác

            # Xử lý cắt giảm
            random.seed(42) # Cố định seed để kết quả đồng nhất
            random.shuffle(exclusive_majority_files)
            num_to_keep = int(len(exclusive_majority_files) * retain_ratio)
            keep_files.extend(exclusive_majority_files[:num_to_keep])

            print(f"--- THỐNG KÊ TẬP TRAIN ---")
            print(f"Tổng số ảnh gốc: {len(label_files)}")
            print(f"Ảnh CHỈ chứa đa số (Class {majority_id}): {len(exclusive_majority_files)}")
            print(f"Đã giữ lại: {num_to_keep} ảnh đa số ({retain_ratio*100}%)")
            print(f"Tổng số ảnh sau cân bằng: {len(keep_files)}\n")

        else:
            # Giữ nguyên 100% dữ liệu cho tập valid và test
            keep_files = label_files
            print(f"Đã copy nguyên vẹn tập {split}: {len(keep_files)} ảnh.")

        # Thực hiện copy các file được giữ lại sang thư mục mới
        for label_file in keep_files:
            # Copy label
            src_lbl = os.path.join(src_labels_dir, label_file)
            tgt_lbl = os.path.join(tgt_labels_dir, label_file)
            shutil.copy(src_lbl, tgt_lbl)

            # Tìm và copy image tương ứng (hỗ trợ .jpg, .png, .jpeg)
            img_name_base = os.path.splitext(label_file)[0]
            for ext in ['.jpg', '.png', '.jpeg', '.JPG']:
                src_img = os.path.join(src_images_dir, img_name_base + ext)
                if os.path.exists(src_img):
                    tgt_img = os.path.join(tgt_images_dir, img_name_base + ext)
                    shutil.copy(src_img, tgt_img)
                    break

# ==========================================
# THỰC THI
# ==========================================
balance_and_preprocess_yolo(SOURCE_DIR, TARGET_DIR, MAJORITY_CLASS_ID, RETAIN_RATIO)
print("\nHoàn tất! Thư mục 'final_dataset' đã sẵn sàng để nén (zip) và upload lên Kaggle.")

--- THỐNG KÊ TẬP TRAIN ---
Tổng số ảnh gốc: 1634
Ảnh CHỈ chứa đa số (Class 1): 341
Đã giữ lại: 102 ảnh đa số (30.0%)
Tổng số ảnh sau cân bằng: 1395

Đã copy nguyên vẹn tập valid: 353 ảnh.

Hoàn tất! Thư mục 'final_dataset' đã sẵn sàng để nén (zip) và upload lên Kaggle.
